# 04 — Advanced Repurchase Prediction with PyTorch MLP

This notebook compares feature sets and imbalance strategies for repurchase prediction.

**Expected input files** (in the repository root, or under `data/`):
- `train_70.csv`
- `val_15.csv`
- `test_15.csv`

Expected core columns: `customer_id`, `Recency`, `Frequency`, `Monetary`, and a binary repurchase label such as `Repurchase_Label`. If your preprocessing notebook uses a different label name, update `TARGET_CANDIDATES` below.

**Leakage safeguards**
- The MLP scaler and K-Means centroids are fitted on the training set only.
- Validation and test rows are transformed using the training-set scaler/centroids.
- SMOTE is applied only to the training set.
- The test set is used only for final evaluation, not model selection or early stopping.

The notebook creates an `output/` folder for CSV summaries, plots, and confusion matrices.

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import copy
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    f1_score, roc_auc_score, accuracy_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
from sklearn.impute import SimpleImputer

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

ROOT = Path(".")
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CANDIDATES = ["train_70.csv", "data/train_70.csv"]
VAL_CANDIDATES   = ["val_15.csv", "data/val_15.csv"]
TEST_CANDIDATES  = ["test_15.csv", "data/test_15.csv"]

TARGET_CANDIDATES = [
    "Repurchase_Label", "repurchase_label", "repurchase", "target",
    "label", "y", "will_repurchase"
]
ID_CANDIDATES = ["customer_id", "CustomerID", "customerID", "user_id", "member_id"]
RFM_CANDIDATES = ["Recency", "Frequency", "Monetary"]

## 2. Load data and identify columns

In [ ]:
def find_file(candidates):
    for item in candidates:
        path = ROOT / item
        if path.exists():
            return path
    raise FileNotFoundError(
        "Could not find any of these files: " + ", ".join(candidates) +
        ". Put the three CSV files in the notebook folder or in data/."
    )

train_path = find_file(TRAIN_CANDIDATES)
val_path = find_file(VAL_CANDIDATES)
test_path = find_file(TEST_CANDIDATES)

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

def find_column(df, candidates, description):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    raise ValueError(
        f"Cannot identify {description}. Available columns: {list(df.columns)}. "
        f"Please add the actual column name to the candidate list."
    )

TARGET_COL = find_column(train_df, TARGET_CANDIDATES, "binary target label")
ID_COL = next((c for c in train_df.columns if str(c).strip().lower() in
               [x.lower() for x in ID_CANDIDATES]), None)

# Standardize RFM column lookup while preserving the original names.
RFM_COLS = []
for name in RFM_CANDIDATES:
    RFM_COLS.append(find_column(train_df, [name], name))

print("Train:", train_df.shape, "Validation:", val_df.shape, "Test:", test_df.shape)
print("Target:", TARGET_COL, "| ID:", ID_COL, "| RFM:", RFM_COLS)
display(train_df.head())
print("\nTarget distribution (train):")
print(train_df[TARGET_COL].value_counts(dropna=False))


## 3. Prepare labels and prevent leakage

In [ ]:
def to_binary_target(series):
    # Supports numeric 0/1 and common string labels.
    if series.isna().any():
        raise ValueError("Target label contains missing values; clean labels in preprocessing first.")
    values = set(pd.Series(series).unique().tolist())
    if values.issubset({0, 1, 0.0, 1.0, False, True}):
        return series.astype(int).to_numpy()
    mapping = {
        "1": 1, "true": 1, "yes": 1, "y": 1, "repurchase": 1, "re-purchase": 1,
        "0": 0, "false": 0, "no": 0, "n": 0, "no repurchase": 0, "not repurchase": 0
    }
    converted = series.astype(str).str.strip().str.lower().map(mapping)
    if converted.isna().any():
        raise ValueError(f"Target values are not recognized as binary 0/1: {sorted(map(str, values))}")
    return converted.astype(int).to_numpy()

y_train = to_binary_target(train_df[TARGET_COL])
y_val = to_binary_target(val_df[TARGET_COL])
y_test = to_binary_target(test_df[TARGET_COL])

# Use only numeric RFM features as the base input; ID and target are never features.
X_train_rfm = train_df[RFM_COLS].apply(pd.to_numeric, errors="coerce")
X_val_rfm = val_df[RFM_COLS].apply(pd.to_numeric, errors="coerce")
X_test_rfm = test_df[RFM_COLS].apply(pd.to_numeric, errors="coerce")

imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train_rfm)
X_val_imp = imputer.transform(X_val_rfm)
X_test_imp = imputer.transform(X_test_rfm)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled = scaler.transform(X_val_imp)
X_test_scaled = scaler.transform(X_test_imp)

print("Positive rate:",
      f"train={y_train.mean():.3f}, val={y_val.mean():.3f}, test={y_test.mean():.3f}")


## 4. Add cluster features: one-hot labels and distances to centroids

In [ ]:
# Fit K-Means on training RFM data only. Never fit centroids on validation/test data.
K = 3
kmeans = KMeans(n_clusters=K, random_state=SEED, n_init=10)
train_cluster = kmeans.fit_predict(X_train_scaled)
val_cluster = kmeans.predict(X_val_scaled)
test_cluster = kmeans.predict(X_test_scaled)

# Distances are calculated to the same training-only centroids for all splits.
train_dist = kmeans.transform(X_train_scaled)
val_dist = kmeans.transform(X_val_scaled)
test_dist = kmeans.transform(X_test_scaled)

def one_hot_clusters(labels, k=K):
    result = np.zeros((len(labels), k), dtype=np.float32)
    result[np.arange(len(labels)), labels] = 1.0
    return result

X_train_onehot = np.concatenate([X_train_scaled, one_hot_clusters(train_cluster)], axis=1)
X_val_onehot = np.concatenate([X_val_scaled, one_hot_clusters(val_cluster)], axis=1)
X_test_onehot = np.concatenate([X_test_scaled, one_hot_clusters(test_cluster)], axis=1)

X_train_distance = np.concatenate([X_train_scaled, train_dist], axis=1)
X_val_distance = np.concatenate([X_val_scaled, val_dist], axis=1)
X_test_distance = np.concatenate([X_test_scaled, test_dist], axis=1)

# Save cluster assignments and distances for inspection.
cluster_output = pd.DataFrame({
    "split": (["train"] * len(y_train)) + (["val"] * len(y_val)) + (["test"] * len(y_test)),
    "cluster": np.concatenate([train_cluster, val_cluster, test_cluster]),
    "distance_cluster_0": np.concatenate([train_dist[:, 0], val_dist[:, 0], test_dist[:, 0]]),
    "distance_cluster_1": np.concatenate([train_dist[:, 1], val_dist[:, 1], test_dist[:, 1]]),
    "distance_cluster_2": np.concatenate([train_dist[:, 2], val_dist[:, 2], test_dist[:, 2]]),
})
if ID_COL and ID_COL in train_df.columns and ID_COL in val_df.columns and ID_COL in test_df.columns:
    cluster_output.insert(0, "customer_id", pd.concat(
        [train_df[ID_COL], val_df[ID_COL], test_df[ID_COL]], ignore_index=True
    ))
cluster_output.to_csv(OUTPUT_DIR / "cluster_distance_features.csv", index=False)

print("Feature dimensions:")
print("RFM:", X_train_scaled.shape[1])
print("RFM + one-hot cluster:", X_train_onehot.shape[1])
print("RFM + centroid distances:", X_train_distance.shape[1])


## 5. Define a small MLP from PyTorch modules

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=32, dropout=0.25):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)  # logits for binary classification
        )

    def forward(self, x):
        return self.network(x).squeeze(1)

def make_class_weights(y):
    counts = np.bincount(y, minlength=2).astype(float)
    weights = len(y) / (2.0 * np.maximum(counts, 1.0))
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def safe_auc(y_true, prob):
    return float(roc_auc_score(y_true, prob)) if len(np.unique(y_true)) == 2 else float("nan")

def evaluate_prob(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    return {
        "F1": f1_score(y_true, pred, zero_division=0),
        "AUC": safe_auc(y_true, prob),
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
    }

def train_mlp(Xtr, ytr, Xv, yv, use_class_weight=False, epochs=150,
              patience=15, lr=1e-3, hidden_dim=32, dropout=0.25, run_name="mlp"):
    Xtr = np.asarray(Xtr, dtype=np.float32)
    Xv = np.asarray(Xv, dtype=np.float32)
    ytr = np.asarray(ytr, dtype=np.float32)
    yv = np.asarray(yv, dtype=np.float32)

    model = MLP(Xtr.shape[1], hidden_dim=hidden_dim, dropout=dropout).to(DEVICE)
    if use_class_weight:
        # BCEWithLogitsLoss uses pos_weight for the positive/minority class.
        negatives = max(int((ytr == 0).sum()), 1)
        positives = max(int((ytr == 1).sum()), 1)
        pos_weight = torch.tensor([negatives / positives], dtype=torch.float32, device=DEVICE)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    train_ds = TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr))
    loader = DataLoader(train_ds, batch_size=128, shuffle=True)

    Xv_tensor = torch.from_numpy(Xv).to(DEVICE)
    yv_tensor = torch.from_numpy(yv).to(DEVICE)
    history = {"train_loss": [], "val_loss": [], "val_f1": []}
    best_state, best_f1, best_epoch, stale = None, -1.0, 0, 0

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_logits = model(Xv_tensor)
            val_loss = criterion(val_logits, yv_tensor).item()
            val_prob = torch.sigmoid(val_logits).cpu().numpy()
        val_metrics = evaluate_prob(yv, val_prob)
        history["train_loss"].append(float(np.mean(losses)))
        history["val_loss"].append(float(val_loss))
        history["val_f1"].append(float(val_metrics["F1"]))

        # Early stopping and model selection use validation F1, not test metrics.
        if val_metrics["F1"] > best_f1:
            best_f1 = val_metrics["F1"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
        if stale >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history, best_epoch

def predict_prob(model, X):
    model.eval()
    with torch.no_grad():
        x = torch.tensor(np.asarray(X, dtype=np.float32), device=DEVICE)
        return torch.sigmoid(model(x)).cpu().numpy()


## 6. Run experiments: feature sets × imbalance strategies

In [ ]:
# If imbalanced-learn is installed, SMOTE is included; otherwise that row is skipped.
try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
except ImportError:
    SMOTE_AVAILABLE = False
    print("imbalanced-learn is not installed; SMOTE experiments will be skipped.")
    print("Install if needed: pip install imbalanced-learn")

feature_sets = {
    "RFM": (X_train_scaled, X_val_scaled, X_test_scaled),
    "RFM + one-hot cluster": (X_train_onehot, X_val_onehot, X_test_onehot),
    "RFM + centroid distances": (X_train_distance, X_val_distance, X_test_distance),
}

strategies = ["raw", "class_weight"] + (["SMOTE"] if SMOTE_AVAILABLE else [])
results = []
histories = {}
trained_models = {}

for feature_name, (Xtr, Xv, Xte) in feature_sets.items():
    for strategy in strategies:
        X_fit, y_fit = Xtr, y_train.copy()
        use_weight = strategy == "class_weight"

        if strategy == "SMOTE":
            # SMOTE touches training data only. Validation/test remain unchanged.
            smote = SMOTE(random_state=SEED)
            X_fit, y_fit = smote.fit_resample(Xtr, y_train)

        run_name = f"{feature_name} | {strategy}"
        print("Training:", run_name)
        model, history, best_epoch = train_mlp(
            X_fit, y_fit, Xv, y_val,
            use_class_weight=use_weight,
            run_name=run_name
        )
        val_prob = predict_prob(model, Xv)
        test_prob = predict_prob(model, Xte)
        val_metrics = evaluate_prob(y_val, val_prob)
        test_metrics = evaluate_prob(y_test, test_prob)

        row = {
            "Model": "MLP (PyTorch scratch)",
            "Feature Set": feature_name,
            "Imbalance Strategy": strategy,
            "Best Epoch": best_epoch,
            **{f"Val {k}": v for k, v in val_metrics.items()},
            **{f"Test {k}": v for k, v in test_metrics.items()},
        }
        results.append(row)
        histories[run_name] = history
        trained_models[run_name] = model

results_df = pd.DataFrame(results).sort_values(
    ["Val F1", "Val AUC"], ascending=False, na_position="last"
)
results_df.to_csv(OUTPUT_DIR / "mlp_results_summary.csv", index=False)
display(results_df)


## 7. Plot training curves and compare F1/AUC

In [ ]:
# Loss curves for every run; validation curves are used to inspect overfitting.
for run_name, history in histories.items():
    plt.figure(figsize=(7, 4))
    plt.plot(history["train_loss"], label="Train loss")
    plt.plot(history["val_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("BCE loss")
    plt.title(f"Training curve: {run_name}")
    plt.legend()
    plt.tight_layout()
    safe_name = "".join(ch if ch.isalnum() else "_" for ch in run_name)
    plt.savefig(OUTPUT_DIR / f"loss_{safe_name}.png", dpi=160)
    plt.show()
    plt.close()

if not results_df.empty:
    plot_df = results_df.copy()
    plot_df["Label"] = plot_df["Feature Set"] + "\n" + plot_df["Imbalance Strategy"]
    x = np.arange(len(plot_df))
    width = 0.38
    plt.figure(figsize=(max(9, len(plot_df) * 1.15), 5))
    plt.bar(x - width/2, plot_df["Test F1"], width, label="Test F1")
    plt.bar(x + width/2, plot_df["Test AUC"], width, label="Test AUC")
    plt.xticks(x, plot_df["Label"], rotation=35, ha="right")
    plt.ylabel("Score")
    plt.title("MLP experiment comparison (final test evaluation)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "mlp_f1_auc_comparison.png", dpi=160)
    plt.show()
    plt.close()


## 8. Confusion matrix and classification report for the selected model

In [ ]:
# Choose the configuration by validation F1; only now report its test performance.
best_row = results_df.iloc[0]
best_name = f'{best_row["Feature Set"]} | {best_row["Imbalance Strategy"]}'
best_model = trained_models[best_name]
best_X_test = feature_sets[best_row["Feature Set"]][2]
best_test_prob = predict_prob(best_model, best_X_test)
best_test_pred = (best_test_prob >= 0.5).astype(int)

print("Selected by validation F1:", best_name)
print("\nTest metrics:")
print(pd.Series(evaluate_prob(y_test, best_test_prob)).to_string())
print("\nClassification report:")
print(classification_report(y_test, best_test_pred, zero_division=0))

cm = confusion_matrix(y_test, best_test_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No repurchase (0)", "Repurchase (1)"])
disp.plot(values_format="d")
plt.title(f"Test confusion matrix: {best_name}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "mlp_best_confusion_matrix.png", dpi=160)
plt.show()
plt.close()

pd.DataFrame({
    "y_true": y_test,
    "y_probability": best_test_prob,
    "y_pred": best_test_pred
}).to_csv(OUTPUT_DIR / "mlp_best_test_predictions.csv", index=False)


## 9. Export a compact report table

In [ ]:
# Keep the principal report metrics compact and easy to paste into the report.
report_table = results_df[[
    "Model", "Feature Set", "Imbalance Strategy",
    "Val F1", "Val AUC", "Test F1", "Test AUC",
    "Test Precision", "Test Recall", "Best Epoch"
]].copy()

report_table.to_csv(OUTPUT_DIR / "mlp_report_table.csv", index=False)
display(report_table)

print(f"\nAll outputs saved to: {OUTPUT_DIR.resolve()}")
print("Important: compare Test F1/AUC descriptively in the report; choose configurations using validation results.")
